# Theme Explorer

A local notebook, not a docs page: the selectors below are JavaScript, which
the built site strips. Run it in Jupyter after `uv sync --group dev`.

Pick a theme and see how it draws every chart, or pick a chart and see it
under every theme. It covers the chart types the theme gallery leaves out,
with values, dense marks and value scales, where the themes differ most.
Re-run the build cell after changing `datachart` to see the current library.


In [ ]:
import base64
import io
import math
import random
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from datachart.charts import (
    BarChart,
    BoxPlot,
    CalendarHeatmap,
    Heatmap,
    Histogram,
    LineChart,
    NetworkChart,
    PyramidChart,
    RadialChart,
    SankeyChart,
    ScatterChart,
    StackedAreaChart,
    Treemap,
    ViolinPlot,
)
from datachart.config import config
from datachart.constants import CALENDAR_WEEKDAY, THEME, VALUE_FORMAT

rng = np.random.RandomState(7)
random.seed(7)
FIGSIZE = (4.6, 3.3)

CATS = ["Bench A", "Bench B", "Bench C", "Bench D"]
GROUPED = [
    [{"label": c, "y": y} for c, y in zip(CATS, ys)]
    for ys in ([66.7, 58.6, 76.7, 83.2], [65.4, 57.7, 77.8, 82.7], [68.5, 53.4, 76.9, 85.9])
]
LINES = [
    [{"x": x, "y": round(40 + 18 * math.sin(x / 3.2) + x * k + rng.uniform(-2, 2), 1)} for x in range(21)]
    for k in (1.6, 0.9, 0.3)
]
SCATTER = [
    [{"x": float(x), "y": float(y)} for x, y in zip(rng.randn(120) + k, rng.randn(120) + k)]
    for k in (0, 1.5, 3)
]
HIST = [{"x": float(v)} for v in rng.randn(300)]
BOX = [
    {"label": g, "value": float(rng.randn() * sd + mu)}
    for g, mu, sd in (("v1.0", 62, 6), ("v1.1", 68, 5), ("v2.0", 74, 7))
    for _ in range(50)
]
HEAT = {"z": [[int(rng.randint(0, 100)) for _ in range(7)] for _ in range(7)]}
NODES = [{"id": f"n{i}", "group": "abcd"[i % 4]} for i in range(40)]
EDGES = [
    {"source": f"n{i}", "target": f"n{rng.randint(40)}", "weight": int(rng.randint(1, 6))}
    for i in range(40)
    for _ in range(3)
]
EDGES = [e for e in EDGES if e["source"] != e["target"]]
TREEMAP = {
    "data": [
        {"label": g, "children": [{"label": n, "value": v} for n, v in rows]}
        for g, rows in {
            "Asia": [("India", 1429), ("China", 1426), ("Indonesia", 278), ("Japan", 123)],
            "Africa": [("Nigeria", 224), ("Ethiopia", 127), ("Egypt", 112)],
            "Europe": [("Russia", 144), ("Germany", 83), ("France", 65)],
        }.items()
    ]
}
SANKEY = {
    "links": [
        {"source": s, "target": t, "value": v}
        for s, t, v in (
            ("Visited", "Signed up", 300),
            ("Visited", "Bounced", 700),
            ("Signed up", "Activated", 180),
            ("Signed up", "Churned", 120),
            ("Activated", "Paid", 90),
            ("Activated", "Free tier", 90),
        )
    ]
}
COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
RADIAL = [{"label": d, "y": v} for d, v in zip(COMPASS, [4, 7, 6, 3, 5, 8, 2, 6])]
PYRAMID = [
    [
        {"label": f"{a}-{a + 9}", "y": float(base - a * 9 + 260 * math.exp(-((a - boom) ** 2) / 300))}
        for a in range(0, 80, 10)
    ]
    for base, boom in ((1500, 30), (1400, 40))
]
STACK = [
    [{"x": i, "y": float(v)} for i, v in enumerate(np.abs(rng.randn(12)).cumsum() + 1)]
    for _ in range(3)
]
DAYS = [date(2024, 1, 1) + timedelta(days=i) for i in range(91)]
CALENDAR = {
    "date": DAYS,
    "value": [int(rng.poisson(3.0 if d.weekday() < 5 else 0.7)) for d in DAYS],
}

# name -> a builder, called with the theme already applied
CHARTS = {
    "Bar": lambda: BarChart(
        GROUPED, subtitle=["one", "two", "three"], show_values=True, show_legend=True, figsize=FIGSIZE
    ),
    "Stacked bar": lambda: BarChart(
        GROUPED, subtitle=["one", "two", "three"], bar_mode="stack", show_values=True, show_legend=True, figsize=FIGSIZE
    ),
    "Line": lambda: LineChart(
        LINES, subtitle=["a", "b", "c"], show_legend=True, figsize=FIGSIZE
    ),
    "Line with values": lambda: LineChart(
        [[p for p in line if p["x"] % 4 == 0] for line in LINES],
        subtitle=["a", "b", "c"], show_values=True, show_legend=True, figsize=FIGSIZE,
    ),
    "Stacked area": lambda: StackedAreaChart(
        STACK, subtitle=["a", "b", "c"], show_legend=True, figsize=FIGSIZE
    ),
    "Scatter": lambda: ScatterChart(
        SCATTER, subtitle=["a", "b", "c"], show_legend=True, figsize=FIGSIZE
    ),
    "Histogram": lambda: Histogram(HIST, num_bins=12, show_values=True, figsize=FIGSIZE),
    "Box plot": lambda: BoxPlot(BOX, show_outliers=True, figsize=FIGSIZE),
    "Violin plot": lambda: ViolinPlot(BOX, figsize=FIGSIZE),
    "Heatmap": lambda: Heatmap(
        HEAT, show_values=True, show_colorbars=True, figsize=FIGSIZE
    ),
    "Calendar heatmap": lambda: CalendarHeatmap(
        CALENDAR, show_values=True, week_start=CALENDAR_WEEKDAY.SUNDAY, figsize=(9, 2.6)
    ),
    "Network": lambda: NetworkChart(
        {"nodes": NODES, "edges": EDGES}, show_legend=True, figsize=FIGSIZE
    ),
    "Network with values": lambda: NetworkChart(
        {"edges": [{"source": s, "target": t, "weight": w} for s, t, w in (
            ("Alpha", "Beta", 8), ("Alpha", "Gamma", 3), ("Beta", "Gamma", 5),
            ("Beta", "Delta", 2), ("Gamma", "Delta", 7), ("Delta", "Alpha", 1),
            ("Gamma", "Epsilon", 4), ("Epsilon", "Beta", 2))]},
        directed=True, show_values=True, figsize=FIGSIZE,
    ),
    "Treemap": lambda: Treemap(
        TREEMAP, show_values=True, value_format=VALUE_FORMAT.INTEGER, figsize=(8, 4.5)
    ),
    "Sankey": lambda: SankeyChart(
        SANKEY, show_values=True, value_format=VALUE_FORMAT.INTEGER, figsize=FIGSIZE
    ),
    "Radial": lambda: RadialChart(RADIAL, mark="bar", show_grid="both", figsize=FIGSIZE),
    "Pyramid": lambda: PyramidChart(
        PYRAMID, subtitle=["Women", "Men"], show_legend=True, figsize=FIGSIZE
    ),
}
THEMES = [
    "default", "material", "minimal", "harbor", "dark", "greyscale", "ink",
    "hatch", "muted", "contrast", "mutedhatch", "slatehatch", "sketch", "quill",
]


The sample data and the builder of each chart are defined in the cell above.
The next cell renders each chart under each theme with
`config.set_theme(...)`, then hands the pictures to a small page with the
selectors.


In [ ]:
def render(build, theme):
    """One chart under one theme, as a WebP data URI."""
    config.set_theme(theme)
    fig = build()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="webp", dpi=100, pil_kwargs={"quality": 82})
    plt.close(fig)
    return "data:image/webp;base64," + base64.b64encode(buffer.getvalue()).decode()


images = {
    theme: {name: render(build, theme) for name, build in CHARTS.items()}
    for theme in THEMES
}
config.reset_config()


In [ ]:
import json

PAGE = """
<style>
.te-bar { display: flex; flex-wrap: wrap; gap: .6em 1.2em; align-items: center; margin: .5em 0 1em; }
.te-bar label { font-size: .8em; display: flex; gap: .4em; align-items: center; }
.te-bar select { font: inherit; padding: .25em .5em; min-width: 11em; }
.te-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(min(100%, 21em), 1fr)); gap: 1em; }
.te-grid figure { margin: 0; }
.te-grid img { width: 100%; height: auto; display: block; background: #fff;
  border: 1px solid rgba(128,128,128,.3); border-radius: .3em; }
.te-grid figcaption { font-size: .8em; margin-top: .3em; opacity: .8; }
.te-grid figcaption code { background: none; padding: 0; }
.te-grid .wide { grid-column: 1 / -1; }
.te-grid .wide img { width: auto; max-width: 100%; }
</style>
<div class="te-bar">
  <label>View
    <select id="te-view"><option value="theme">One theme, every chart</option>
    <option value="chart">One chart, every theme</option></select></label>
  <label>Theme <select id="te-theme"></select></label>
  <label>Chart <select id="te-chart"></select></label>
</div>
<div class="te-grid" id="te-grid"></div>
<script>
(function () {
  var DATA = __DATA__;
  var themes = Object.keys(DATA), charts = Object.keys(DATA[themes[0]]);
  var $ = function (id) { return document.getElementById(id); };
  var view = $("te-view"), theme = $("te-theme"), chart = $("te-chart"), grid = $("te-grid");
  themes.forEach(function (t) { theme.add(new Option(t, t)); });
  charts.forEach(function (c) { chart.add(new Option(c, c)); });
  function card(src, caption, wide) {
    var f = document.createElement("figure"), i = new Image(), c = document.createElement("figcaption");
    if (wide) f.className = "wide";
    i.src = src; i.alt = caption; c.textContent = caption;
    f.appendChild(i); f.appendChild(c); return f;
  }
  function draw() {
    var byTheme = view.value === "theme";
    theme.disabled = !byTheme; chart.disabled = byTheme;
    grid.replaceChildren.apply(grid, (byTheme ? charts : themes).map(function (k) {
      return byTheme
        ? card(DATA[theme.value][k], k, k === "Calendar heatmap")
        : card(DATA[k][chart.value], k, chart.value === "Calendar heatmap");
    }));
  }
  [view, theme, chart].forEach(function (s) { s.addEventListener("change", draw); });
  draw();
})();
</script>
"""

display(HTML(PAGE.replace("__DATA__", json.dumps(images))))


Applying a theme replaces the whole global configuration, so the last line of the
build cell resets it.
